# Stage 5 — Personalized-Window Extraction

**The idea:** the current epochs file uses ONE global window (300ms, ending
150ms pre-response) for all 24 subjects, chosen to fit the FASTEST subject
(sub-04). But 21/24 subjects could afford a longer window -- some up to
800ms -- while still keeping >=95% of their trials, since they simply
respond slower.

**Personalized window plan (computed from RT data alone, no labels touched):**

```
sub-01: 350ms
sub-02: 700ms
sub-03: 600ms
sub-04: 250ms
sub-05: 600ms
sub-06: 500ms
sub-07: 400ms
sub-08: 700ms
sub-09: 500ms
sub-10: 800ms
sub-11: 250ms
sub-12: 600ms
sub-13: 500ms
sub-14: 750ms
sub-15: 800ms
sub-16: 650ms
sub-17: 400ms
sub-18: 650ms
sub-19: 700ms
sub-20: 700ms
sub-21: 700ms
sub-22: 300ms
sub-23: 700ms
sub-24: 600ms
```

**What this produces:** since window lengths now differ across subjects,
epochs can't be stacked into one uniform array like before. This notebook
saves each subject's epochs at their OWN length, padded with NaN to the
maximum length (800ms = 200 samples @ 250Hz) so everything still fits in
one file; a subject's true window is recoverable from `window_len_ms` per
subject, and any NaN padding should be excluded from that subject's
features at analysis time.

Run this once; it produces `eeg_epochs_personalized.npz`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!apt-get update -qq && apt-get install -y octave -qq
print("ready")

In [ ]:
DATA_DIR = "/content/drive/MyDrive/PATH_TO_YOUR_EEG_FOLDER"   # <-- EDIT
OUT_PATH = "/content/drive/MyDrive/eeg_epochs_personalized.npz"

BUFFER_MS = 150
TARGET_FS = 250
ORIG_FS = 1000
DS = ORIG_FS // TARGET_FS

# Personalized window length per subject (ms), computed from RT feasibility
# at >=95% retention. Fast-responding subjects keep short windows; slow
# responders get longer ones. This does NOT use any classification label.
PERSONAL_WINDOW_MS = {'01': 350, '02': 700, '03': 600, '04': 250, '05': 600, '06': 500, '07': 400, '08': 700, '09': 500, '10': 800, '11': 250, '12': 600, '13': 500, '14': 750, '15': 800, '16': 650, '17': 400, '18': 650, '19': 700, '20': 700, '21': 700, '22': 300, '23': 700, '24': 600}

MAX_WINDOW_MS = max(PERSONAL_WINDOW_MS.values())
MAX_SAMPLES_OUT = int(MAX_WINDOW_MS / 1000 * TARGET_FS)
print(f"Max window across subjects: {MAX_WINDOW_MS}ms -> {MAX_SAMPLES_OUT} samples at {TARGET_FS}Hz")
print(f"Shorter-window subjects will be NaN-padded at the START (earliest part missing)")

In [ ]:
import os, glob, re, subprocess, gc, time
import numpy as np
from scipy.io import loadmat

pat = re.compile(r"sub-(\d+)_run-(\d+)")
def key_of(p):
    m = pat.search(os.path.basename(p)); return (m.group(1), m.group(2)) if m else None
dmap = {key_of(f): f for f in glob.glob(os.path.join(DATA_DIR,"**","EEG_data_sub-*.mat"), recursive=True) if key_of(f)}
emap = {key_of(f): f for f in glob.glob(os.path.join(DATA_DIR,"**","EEG_events_sub-*.mat"), recursive=True) if key_of(f)}
paired = sorted(set(dmap) & set(emap))
print(f"{len(paired)} subject/run pairs")

In [ ]:
def load_eeg_via_octave(mat_path, tmp_out="/content/tmp_Y.mat"):
    if os.path.exists(tmp_out): os.remove(tmp_out)
    cmd = (f"load('{mat_path}', 'EEGdata'); Y=EEGdata.Y; "
           f"save('-mat7-binary', '{tmp_out}', 'Y');")
    r = subprocess.run(["octave","--no-gui","--quiet","--eval",cmd],
                       capture_output=True, text=True)
    if not os.path.exists(tmp_out):
        raise RuntimeError(f"Octave failed for {mat_path}:\n{r.stderr[:500]}")
    Y = loadmat(tmp_out)["Y"]
    os.remove(tmp_out)
    return Y

In [ ]:
# MAIN LOOP -- each subject uses ITS OWN window length, right-aligned
# (window always ENDS at response-150ms; shorter windows simply start later,
# so all windows share the same right edge and only differ in how far back
# they reach). Output is padded to MAX_SAMPLES_OUT with NaN at the START
# for subjects with a shorter personal window.

X_all, subj_all, run_all = [], [], []
conf_all, acc_all, choice_all, dotdir_all, rt_all, tidx_all = [], [], [], [], [], []
win_len_all = []
log = []
t0 = time.time()

for i, key in enumerate(paired):
    subj, run = key
    win_ms = PERSONAL_WINDOW_MS.get(subj, 300)
    win_samples_native = int(win_ms / 1000 * ORIG_FS)   # at 1000Hz, before downsampling
    n_out_this = int(win_ms / 1000 * TARGET_FS)

    print(f"[{i+1}/{len(paired)}] sub-{subj} run-{run} (personal window={win_ms}ms) ...", end=" ", flush=True)
    try:
        ev = loadmat(emap[key])
        tstim = ev["tstim"].squeeze().astype(float)
        RT    = ev["RT"].squeeze().astype(float)
        confidence = ev["confidence"].squeeze().astype(int)
        accuracy   = ev["accuracy"].squeeze().astype(int)
        choice     = ev["choice"].squeeze().astype(int)
        dotdir     = ev["dotdirection"].squeeze().astype(int)

        EEG = load_eeg_via_octave(dmap[key])
        n_ch, n_samp = EEG.shape

        n_kept = n_skip = 0
        for t in range(len(tstim)):
            resp_time = tstim[t] + RT[t]
            win_end   = int(round(resp_time - BUFFER_MS))
            win_start = win_end - win_samples_native
            if win_start < 0 or win_end > n_samp:
                n_skip += 1; continue
            seg = EEG[:, win_start:win_end]
            if seg.shape[1] != win_samples_native:
                n_skip += 1; continue
            seg = seg[:, :DS*n_out_this].reshape(n_ch, n_out_this, DS).mean(axis=2)

            # place into a MAX_SAMPLES_OUT canvas, right-aligned (pad NaN at start)
            canvas = np.full((n_ch, MAX_SAMPLES_OUT), np.nan, dtype=np.float32)
            canvas[:, MAX_SAMPLES_OUT - n_out_this:] = seg.astype(np.float32)

            X_all.append(canvas)
            subj_all.append(subj); run_all.append(run)
            conf_all.append(confidence[t]); acc_all.append(accuracy[t])
            choice_all.append(choice[t]); dotdir_all.append(dotdir[t])
            rt_all.append(RT[t]); tidx_all.append(t); win_len_all.append(win_ms)
            n_kept += 1

        log.append((subj, run, len(tstim), n_kept, n_skip, win_ms))
        print(f"kept {n_kept}/{len(tstim)}")
        del EEG; gc.collect()
    except Exception as e:
        print(f"FAILED: {e}")
        log.append((subj, run, -1, 0, -1, win_ms))

print(f"\nElapsed {(time.time()-t0)/60:.1f} min; epochs: {len(X_all)}")

In [ ]:
X = np.stack(X_all)
print("X shape:", X.shape, f"({X.nbytes/1e6:.0f} MB in memory)")
print(f"NaN fraction overall: {np.isnan(X).mean()*100:.1f}% "
      f"(expected -- shorter-window subjects are padded)")

np.savez_compressed(
    OUT_PATH, X=X,
    subject=np.array(subj_all), run=np.array(run_all),
    confidence=np.array(conf_all), accuracy=np.array(acc_all),
    choice=np.array(choice_all), dotdirection=np.array(dotdir_all),
    RT=np.array(rt_all), trial_index=np.array(tidx_all),
    window_len_ms_per_trial=np.array(win_len_all),
    max_window_ms=MAX_WINDOW_MS, buffer_ms=BUFFER_MS, fs=TARGET_FS,
    personal_window_plan=str(PERSONAL_WINDOW_MS),
)
print(f"Saved {OUT_PATH} ({os.path.getsize(OUT_PATH)/1e6:.1f} MB)")

In [ ]:
# Report
print(f"{'sub':>5} {'run':>4} {'total':>6} {'kept':>5} {'skip':>5} {'win_ms':>7}")
for row in log:
    print("{:>5} {:>4} {:>6} {:>5} {:>5} {:>7}".format(*row))

## What to send back

Just upload `eeg_epochs_personalized.npz` (should be a few hundred MB --
if over 480MB, tell me and I'll adjust TARGET_FS down to 200).
I'll run the comparison against the fixed-window baseline myself.